In [2]:
!pip install sqlalchemy tqdm transformers torch torchvision torchaudio


  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-manylinu

In [3]:
%%time

import warnings
warnings.filterwarnings("ignore")

from concurrent.futures import ThreadPoolExecutor
import multiprocessing
from tqdm.notebook import tqdm
from PIL import Image
import requests
import io
import pandas as pd
import torch
import time

from google.colab import drive

import os

# if not mounted
if not os.path.exists('/content/drive'):
  drive.mount('/content/drive')

from sqlalchemy import create_engine

# Create a sqlite engine instance
engine = create_engine(f"sqlite:///drive/MyDrive/colab/banshee/data/nfs_m.db")

import pandas as pd

# Read the database as a dataframe
raw_df = pd.read_sql_table('video_statistics', engine)

import os

image_dir = '/content/drive/MyDrive/colab/banshee/data/images'
os.makedirs(image_dir, exist_ok=True)

# Filter out already downloaded images
filenames = raw_df['video_id'] + '.jpg'
video_statistics_filenames = set(filenames.tolist())
saved_filenames = set(os.listdir(image_dir))
filtered_video_ids = list(set(video_statistics_filenames) - saved_filenames)

len(filtered_video_ids)

CPU times: user 20.1 s, sys: 5.19 s, total: 25.3 s
Wall time: 36.6 s


1150311

In [1]:
import nest_asyncio
import asyncio
import aiohttp
import os
from tqdm import tqdm

# Function to download image with concurrency control
async def download_image(session, video_id, progress_bar, image_dir, semaphore):
    async with semaphore:
        base_url = f"https://img.youtube.com/vi/{video_id}/"
        resolutions = ["maxresdefault.jpg", "hqdefault.jpg", "mqdefault.jpg", "default.jpg"]
        for res in resolutions:
            url = base_url + res
            print(url)
            async with session.get(url) as response:
                if response.status == 200:
                    content = await response.read()
                    # Save the image
                    save_path = os.path.join(image_dir, video_id)
                    print(save_path)
                    with open(save_path, 'wb') as file:
                        file.write(content)
                    progress_bar.update(1)
                    return
        # If none of the resolutions are available
        # print(f"Failed to download image for video ID: {video_id}")
        progress_bar.update(1)

# Main function to manage the download process
async def main(video_ids, image_dir):
    progress_bar = tqdm(total=len(video_ids), desc="Downloading images")
    semaphore = asyncio.Semaphore(10)  # Limit concurrent downloads to 10
    async with aiohttp.ClientSession() as session:
        tasks = [download_image(session, video_id, progress_bar, image_dir, semaphore) for video_id in video_ids]
        await asyncio.gather(*tasks)

    progress_bar.close()

# Necessary setup for running asyncio in Jupyter Notebook
nest_asyncio.apply()

# Execute the asynchronous download if there are video IDs to process
if filtered_video_ids:
    asyncio.get_event_loop().run_until_complete(main(filtered_video_ids, image_dir))
else:
    print("All images already downloaded")


NameError: name 'filtered_video_ids' is not defined